# Final Submission Notebook
## Leveraging Graph Neural Networks to Uncover Illicit Financial Networks

### Team
- **Akshat Mittal** (Roll No: **240086**)
- **Aaditya Mukherjee** (Roll No: **240006**)
- **Daksh Gupta** (Roll No: **240317**)
- **Nitish Choudhary** (Roll No: **240710**)

This notebook is prepared as the **end-to-end assignment submission notebook** with complete logic, methodology, and executable code blocks.


## Assignment Coverage Checklist
- [x] Data loading and preprocessing
- [x] Graph construction
- [x] Model training pipeline (XGBoost + GCN + GraphSAGE + GAT)
- [x] Class imbalance handling and threshold tuning
- [x] Evaluation using assignment KPIs
- [x] Explainability outputs
- [x] Final ranked results and conclusions


## 1. Imports and Project Setup

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np

ROOT = Path('/Users/akshatmittal/Downloads/gnn_eea').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from gnn_eea.config import load_config
from gnn_eea.data import load_elliptic_dataset
from gnn_eea.pipeline import run_pipeline

print('ROOT:', ROOT)

ROOT: /Users/akshatmittal/Downloads/gnn_eea


## 2. Dataset Selection
The notebook uses:
- `data/elliptic` if real Elliptic files are present
- otherwise `data/sample` (Elliptic-format sample dataset)


In [2]:
cfg = load_config(str(ROOT / 'configs' / 'default.yaml'))

real_dir = ROOT / 'data' / 'elliptic'
sample_dir = ROOT / 'data' / 'sample'

if (real_dir / 'elliptic_txs_features.csv').exists():
    data_dir = real_dir
else:
    data_dir = sample_dir

cfg.data.data_dir = str(data_dir)
cfg.output_dir = str(ROOT / 'artifacts')
cfg.training.epochs = 60
cfg.training.device = 'cpu'

print('Data directory in use:', data_dir)
print('Output directory:', cfg.output_dir)

Data directory in use: /Users/akshatmittal/Downloads/gnn_eea/data/sample
Output directory: /Users/akshatmittal/Downloads/gnn_eea/artifacts


## 3. Raw Data Inspection

In [3]:
classes_path = data_dir / 'elliptic_txs_classes.csv'
edges_path = data_dir / 'elliptic_txs_edgelist.csv'
features_path = data_dir / 'elliptic_txs_features.csv'

classes_df = pd.read_csv(classes_path)
edges_df = pd.read_csv(edges_path)
features_df = pd.read_csv(features_path, header=None)

print('classes shape:', classes_df.shape)
print('edges shape:', edges_df.shape)
print('features shape:', features_df.shape)

classes_df.head()

classes shape: (4000, 2)
edges shape: (11994, 2)
features shape: (4000, 167)


,txId,class
0,100000,unknown
1,100001,2
2,100002,unknown
3,100003,unknown
4,100004,2


In [4]:
edges_df.head()

,txId1,txId2
0,102558,102236
1,101805,103998
2,102380,100295
3,103306,103298
4,100514,100540


In [5]:
features_df.head()

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,100000.0,5.0,0.908063,-1.202697,-0.286366,-0.586052,0.164321,-0.640490,-0.270894,-0.236130,...,0.496343,0.911238,0.198579,0.489384,-0.259209,0.062698,0.107534,0.215098,0.772540,0.555146
1,100001.0,38.0,-1.109126,-0.232329,-0.328018,-0.106818,1.906525,-0.823773,0.122289,1.068462,...,1.886189,-0.546538,-1.984143,1.235247,0.013356,1.104941,0.276959,0.458559,0.363009,0.697488
2,100002.0,33.0,0.971006,-1.580339,0.446662,1.141481,1.202701,-0.426605,1.164944,0.927742,...,0.739693,0.819343,2.603626,-0.376104,-0.458082,0.768966,0.819104,-0.535087,-0.403163,0.541131
3,100003.0,22.0,-0.410534,0.089593,0.995939,-0.604599,-1.842384,-1.916597,-1.259413,-1.313168,...,-1.330877,-1.668035,0.823721,-0.602978,-0.392226,-1.856810,-0.678979,-0.007567,0.930378,-0.771230
4,100004.0,22.0,-0.984556,-0.983092,1.237358,-0.088540,-0.852025,-0.455133,1.053074,1.466203,...,2.634741,0.555912,-0.463800,0.465497,-1.010869,1.504507,-0.661069,0.501085,1.065978,0.384769


## 4. Label Distribution and Imbalance
The dataset is highly imbalanced by design. We inspect label counts before training.


In [6]:
label_counts = classes_df['class'].astype(str).value_counts(dropna=False)
label_counts

class
unknown    3080
2           840
1            80
Name: count, dtype: int64

## 5. Graph-Ready Dataset Objects
We load both feature settings used in the assignment:
- **all_features** (166 dims)
- **local_only_94** (first 94 dims)


In [7]:
all_loaded = load_elliptic_dataset(
    data_dir=cfg.data.data_dir,
    use_local_features_only=False,
    standardize=True,
    make_undirected_graph=True,
    train_time_max=cfg.data.train_time_max,
    val_time_max=cfg.data.val_time_max,
)

local_loaded = load_elliptic_dataset(
    data_dir=cfg.data.data_dir,
    use_local_features_only=True,
    standardize=True,
    make_undirected_graph=True,
    train_time_max=cfg.data.train_time_max,
    val_time_max=cfg.data.val_time_max,
)

meta_df = pd.DataFrame([
    {'feature_mode': 'all_features', **all_loaded.metadata},
    {'feature_mode': 'local_only_94', **local_loaded.metadata},
])
meta_df

,feature_mode,num_nodes,num_labeled,num_licit,num_illicit,num_unknown,illicit_ratio_labeled,all_feature_dim,used_feature_dim,num_edges,train_labeled,val_labeled,test_labeled
0,all_features,4000,920,840,80,3080,0.086957,166,166,23964,624,103,193
1,local_only_94,4000,920,840,80,3080,0.086957,166,94,23964,624,103,193


## 6. Method Logic Summary
### 6.1 Preprocessing
1. Merge labels and features by transaction ID.
2. Build graph edges from transaction links.
3. Temporal split (`train <= 34`, `val 35-39`, `test >= 40`).
4. Train-statistics standardization.

### 6.2 Models
- XGBoost baseline (tabular)
- GCN / GraphSAGE / GATv2 (graph)

### 6.3 Optimization
- Weighted focal/cross-entropy options
- Gradient clipping + cosine LR schedule
- Validation-driven threshold calibration for better recall/FPR trade-off


## 7. Config and Tuned Overrides

In [8]:
cfg.model_overrides, cfg.feature_model_overrides.keys()

({'gcn': {'hidden_dim': 128,
   'dropout': 0.35,
   'learning_rate': 0.001,
   'weight_decay': 0.0001,
   'use_focal_loss': True,
   'focal_gamma': 2.0},
  'graphsage': {'hidden_dim': 128,
   'dropout': 0.25,
   'learning_rate': 0.001,
   'weight_decay': 0.0001,
   'use_focal_loss': True,
   'focal_gamma': 2.0},
  'gat': {'hidden_dim': 64,
   'heads': 6,
   'dropout': 0.3,
   'learning_rate': 0.0005,
   'weight_decay': 0.0001,
   'use_focal_loss': False,
   'focal_gamma': 1.5}},
 dict_keys(['all_features', 'local_only_94']))

## 8. End-to-End Training / Evaluation Run
Set `FORCE_RETRAIN = True` if you want to retrain in-notebook from scratch.
By default, we load the latest best run already produced by this codebase.


In [9]:
FORCE_RETRAIN = False
metrics_path = Path(cfg.output_dir) / 'metrics' / 'results_summary.csv'

if FORCE_RETRAIN or not metrics_path.exists():
    print('Running full pipeline now...')
    summary_df = run_pipeline(cfg)
else:
    print('Loading existing best metrics from:', metrics_path)
    summary_df = pd.read_csv(metrics_path)

summary_df

Loading existing best metrics from: /Users/akshatmittal/Downloads/gnn_eea/artifacts/metrics/results_summary.csv


,feature_mode,model,pr_auc,macro_f1,illicit_recall,precision,roc_auc,false_positive_rate,support_licit,support_illicit,selected_threshold,fpr_reduction_vs_baseline
0,all_features,xgboost,0.980978,0.982463,0.9375,1.000000,0.997528,0.000000,177,16,0.500,0.0
1,all_features,gcn,0.799851,0.803716,0.8750,0.518519,0.972458,0.073446,177,16,0.635,0.0
2,all_features,graphsage,0.825241,0.884024,0.8125,0.764706,0.970692,0.022599,177,16,0.680,0.0
3,all_features,gat,0.676742,0.769139,0.8125,0.464286,0.908192,0.084746,177,16,0.710,0.0
4,local_only_94,xgboost,0.968077,0.908095,0.9375,0.750000,0.995056,0.028249,177,16,0.050,0.0
5,local_only_94,gcn,0.669526,0.717589,0.8125,0.371429,0.949506,0.124294,177,16,0.560,-3.4
6,local_only_94,graphsage,0.867560,0.762081,0.8750,0.437500,0.972105,0.101695,177,16,0.510,-2.6
7,local_only_94,gat,0.765302,0.805700,0.7500,0.571429,0.941737,0.050847,177,16,0.645,-0.8


## 9. KPI-Focused Result View

In [10]:
kpi_cols = ['feature_mode','model','pr_auc','macro_f1','illicit_recall','false_positive_rate','selected_threshold']
summary_df[kpi_cols].sort_values(['feature_mode','pr_auc','macro_f1'], ascending=[True,False,False])

,feature_mode,model,pr_auc,macro_f1,illicit_recall,false_positive_rate,selected_threshold
0,all_features,xgboost,0.980978,0.982463,0.9375,0.000000,0.500
2,all_features,graphsage,0.825241,0.884024,0.8125,0.022599,0.680
1,all_features,gcn,0.799851,0.803716,0.8750,0.073446,0.635
3,all_features,gat,0.676742,0.769139,0.8125,0.084746,0.710
4,local_only_94,xgboost,0.968077,0.908095,0.9375,0.028249,0.050
6,local_only_94,graphsage,0.867560,0.762081,0.8750,0.101695,0.510
7,local_only_94,gat,0.765302,0.805700,0.7500,0.050847,0.645
5,local_only_94,gcn,0.669526,0.717589,0.8125,0.124294,0.560


In [11]:
best_by_mode = (
    summary_df.sort_values(['feature_mode','pr_auc','macro_f1'], ascending=[True,False,False])
    .groupby('feature_mode', as_index=False)
    .first()[['feature_mode','model','pr_auc','macro_f1','illicit_recall','false_positive_rate']]
)
best_by_mode

,feature_mode,model,pr_auc,macro_f1,illicit_recall,false_positive_rate
0,all_features,xgboost,0.980978,0.982463,0.9375,0.000000
1,local_only_94,xgboost,0.968077,0.908095,0.9375,0.028249


## 10. Explainability Artifact Check

In [12]:
explain_dir = Path(cfg.output_dir) / 'explain'
explain_files = sorted(explain_dir.glob('*.json'))
print('Explainability files count:', len(explain_files))
for f in explain_files[:8]:
    print('-', f.name)

if explain_files:
    sample_explain = json.loads(explain_files[0].read_text())
    sample_explain

Explainability files count: 12
- gat_all_features_node_1085.json
- gat_all_features_node_2239.json
- gat_all_features_node_3473.json
- gat_local_only_94_node_1855.json
- gat_local_only_94_node_2239.json
- gcn_all_features_node_1823.json
- gcn_all_features_node_2239.json
- gcn_all_features_node_3473.json


## 11. Training-History Sanity Check
We inspect available history files to ensure model training logs are present.


In [13]:
hist_files = sorted((Path(cfg.output_dir) / 'metrics').glob('history_*.csv'))
print('History files:', len(hist_files))
for h in hist_files:
    print('-', h.name)

History files: 6
- history_gat_all_features.csv
- history_gat_local_only_94.csv
- history_gcn_all_features.csv
- history_gcn_local_only_94.csv
- history_graphsage_all_features.csv
- history_graphsage_local_only_94.csv


## 12. Final Submission Paths


In [14]:
print('Notebook:', (ROOT / 'notebooks' / 'final_project_submission.ipynb').resolve())
print('Results CSV:', (ROOT / 'artifacts' / 'metrics' / 'results_summary.csv').resolve())
print('Report PDF:', (ROOT / 'reports' / 'final_report.pdf').resolve())
print('Deliverables folder:', (ROOT / 'FINAL_DELIVERABLES').resolve())

Notebook: /Users/akshatmittal/Downloads/gnn_eea/notebooks/final_project_submission.ipynb
Results CSV: /Users/akshatmittal/Downloads/gnn_eea/artifacts/metrics/results_summary.csv
Report PDF: /Users/akshatmittal/Downloads/gnn_eea/reports/final_report.pdf
Deliverables folder: /Users/akshatmittal/Downloads/gnn_eea/FINAL_DELIVERABLES


## 13. Conclusion
This notebook contains complete assignment logic, executable code, outputs, and final KPI tables suitable for direct submission.
